<a href="https://colab.research.google.com/github/marcouras/AI-engineering-fundamentals/blob/main/lezione3/Lezione3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# 🤖 AI Engineering Fundamentals
## Lezione 3 — Conversazione & Memoria

**ITS Novitas 4.0 — Sviluppatore Intelligenza Artificiale**  
Docente: Marco Uras | 📅 Martedì 26/05/2026

---

### 🎯 Obiettivi
- ✅ Gestire la conversation history (multi-turno)
- ✅ Implementare troncamento e sliding window
- ✅ Aggiungere lo streaming delle risposte
- ✅ Salvare e ricaricare la memoria su file JSON

In [ ]:
# Setup
!pip install anthropic -q
from google.colab import userdata
import anthropic, os, json

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
print("✅ Setup completato!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 29.3 MB/s eta 0:00:00
✅ Setup completato!


---
## 1. Conversation History — il chatbot che ricorda

Il modello **non ha memoria propria**. Per farlo 'ricordare', dobbiamo inviargli tutta la conversazione ad ogni chiamata.

In [ ]:
# Chatbot multi-turno base
history = []

def chat(messaggio, system=None):
    """Invia un messaggio mantenendo la history."""
    # 1. Aggiungi il messaggio dell'utente
    history.append({"role": "user", "content": messaggio})

    # 2. Invia TUTTA la history al modello
    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 500,
        "messages": history
    }
    if system:
        params["system"] = system

    risposta = client.messages.create(**params)
    testo = risposta.content[0].text

    # 3. Aggiungi la risposta alla history
    history.append({"role": "assistant", "content": testo})

    return testo

print("✅ Funzione chat pronta!")

✅ Funzione chat pronta!


In [ ]:
# Proviamo la memoria!
history = []  # Reset

print("👤 Mi chiamo Marco e sono di Sassari.")
r1 = chat("Mi chiamo Marco e sono di Sassari.")
print(f"🤖 {r1}\n")

print("👤 Qual è la capitale della Sardegna?")
r2 = chat("Qual è la capitale della Sardegna?")
print(f"🤖 {r2}\n")

print("👤 Come mi chiamo?")
r3 = chat("Come mi chiamo?")  # Ricorda il nome?
print(f"🤖 {r3}\n")

print(f"📊 Messaggi in history: {len(history)}")

👤 Mi chiamo Marco e sono di Sassari.
🤖 Piacere, Marco! Sassari è una bellissima città in Sardegna, con una ricca storia e cultura. 

Sono qui per aiutarti con domande, conversazioni, informazioni o qualsiasi cosa tu abbia bisogno. C'è qualcosa di particolare con cui posso assisterti?

👤 Qual è la capitale della Sardegna?
🤖 La capitale della Sardegna è **Cagliari**. 

È la città più grande dell'isola e si trova nella parte meridionale. Cagliari è un importante centro politico, economico e culturale della regione, con un porto storico e molti siti di interesse archeologico e architettonico.

👤 Come mi chiamo?
🤖 Ti chiami **Marco**! Me l'hai detto all'inizio della nostra conversazione.

📊 Messaggi in history: 6


In [ ]:
# Vediamo quanti token stiamo usando
from anthropic import Anthropic

count = client.messages.count_tokens(
    model="claude-haiku-4-5-20251001",
    messages=history
)
print(f"📊 Token nella history attuale: {count.input_tokens}")
print(f"💰 Costo stimato prossima chiamata: ${count.input_tokens / 1_000_000 * 1.0:.6f}")
print()
print("💡 Nota: la history cresce ad ogni messaggio — dobbiamo gestirla!")

📊 Token nella history attuale: 230
💰 Costo stimato prossima chiamata: $0.000230

💡 Nota: la history cresce ad ogni messaggio — dobbiamo gestirla!


---
## 2. Gestire la Context Window

Tre strategie per evitare che la history cresca all'infinito.

In [ ]:
# STRATEGIA 1: Truncation
MAX_MESSAGGI = 6  # massimo 3 turni (user + assistant)

def chat_con_troncamento(messaggio, system=None):
    history.append({"role": "user", "content": messaggio})

    # Tronca se troppo lunga
    if len(history) > MAX_MESSAGGI:
        history[:] = history[-MAX_MESSAGGI:]
        print(f"  ✂️  History troncata a {MAX_MESSAGGI} messaggi")

    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=history
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    return testo

# Test
history = []
for i in range(5):
    r = chat_con_troncamento(f"Messaggio numero {i+1}. Quanti messaggi ricordi?")
    print(f"Turno {i+1} | History: {len(history)} msg | Risposta: {r[:80]}...")

Turno 1 | History: 2 msg | Risposta: Ricordo **1 messaggio**: questo che hai appena scritto.

Non ho accesso ai messa...
Turno 2 | History: 4 msg | Risposta: Ricordo **2 messaggi**:

1. "Quanti messaggi ricordi?" (il tuo primo messaggio)
...
Turno 3 | History: 6 msg | Risposta: Ricordo **3 messaggi**:

1. "Quanti messaggi ricordi?" (il tuo primo messaggio)
...
  ✂️  History troncata a 6 messaggi
Turno 4 | History: 7 msg | Risposta: Ricordo **4 messaggi**:

1. "Quanti messaggi ricordi?" (il tuo primo messaggio)
...
  ✂️  History troncata a 6 messaggi
Turno 5 | History: 7 msg | Risposta: Ricordo **5 messaggi**:

1. "Quanti messaggi ricordi?" (il tuo primo messaggio)
...


In [ ]:
# STREAMING — output token per token
print("🌊 Risposta in streaming:\n")

full_text = ""
with client.messages.stream(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    messages=[{"role": "user", "content": "Raccontami in 3 frasi cos'è il machine learning."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
        full_text += text

print(f"\n\n📊 Testo totale: {len(full_text)} caratteri")

🌊 Risposta in streaming:

# Machine Learning

Il machine learning è una branca dell'intelligenza artificiale che permette ai computer di imparare dai dati senza essere esplicitamente programmati per ogni situazione. Gli algoritmi individuano pattern e regolarità nei dati per fare previsioni o decisioni su nuovi casi simili. È la tecnologia dietro i suggerimenti di Netflix, i filtri anti-spam e il riconoscimento facciale dei nostri smartphone.

📊 Testo totale: 420 caratteri


---
## 3. Memoria Persistente su JSON

Salviamo la history su file — il chatbot ricorda tra sessioni diverse.

In [ ]:
import json, os

MEMORY_FILE = "chat_history.json"

def carica_storia():
    """Carica la history dal file JSON. Restituisce lista vuota se non esiste."""
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            storia = json.load(f)
            print(f"📂 Storia caricata: {len(storia)} messaggi precedenti")
            return storia
    print("🆕 Nessuna storia precedente — nuova conversazione")
    return []

def salva_storia(history):
    """Salva la history su file JSON."""
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    print(f"💾 Storia salvata: {len(history)} messaggi")

def chat_persistente(messaggio, system=None):
    """Chatbot con memoria persistente tra sessioni."""
    history.append({"role": "user", "content": messaggio})

    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 400,
        "messages": history
    }
    if system:
        params["system"] = system

    risposta = client.messages.create(**params)
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})

    salva_storia(history)  # Salva dopo ogni messaggio
    return testo

print("✅ Funzioni di persistenza pronte!")

✅ Funzioni di persistenza pronte!


In [ ]:
# Simulazione sessione 1
print("=" * 50)
print("SESSIONE 1")
print("=" * 50)

history = carica_storia()  # Carica eventuali messaggi precedenti

r = chat_persistente("Ciao! Mi chiamo Luca e studio AI engineering a Sassari.")
print(f"🤖 {r}\n")

r = chat_persistente("Qual è la lezione più difficile secondo te?")
print(f"🤖 {r}\n")

print("\n--- Fine sessione 1 ---")
print(f"History salvata con {len(history)} messaggi")

SESSIONE 1
🆕 Nessuna storia precedente — nuova conversazione
💾 Storia salvata: 2 messaggi
🤖 Ciao Luca! 👋 Piacere di conoscerti!

Che interessante - studi AI engineering a Sassari! È un campo affascinante e in crescita. Mi piacerebbe saperne di più:

- Che anno sei?
- Quali aspetti dell'AI ti interessano di più? (machine learning, NLP, computer vision, ecc.)
- Ci sono progetti specifici su cui stai lavorando?

Sono qui se hai domande su argomenti di studio, progetti, o semplicemente se vuoi scambiare due chiacchiere! 😊

💾 Storia salvata: 4 messaggi
🤖 Buona domanda! Secondo me, le parti più impegnative dell'AI engineering di solito sono:

1. **Matematica sottostante** - Calcolo, algebra lineare, probabilità e statistiche. Non è solo memorizzare, ma *veramente capire* come e perché funzionano gli algoritmi.

2. **Il gap teoria-pratica** - Leggere di una rete neurale è diverso da implementarla bene. I dettagli implementativi (normalizzazione dati, hyperparameter tuning, debugging modelli) 

In [ ]:
# Simulazione sessione 2 — ricarica la memoria!
print("=" * 50)
print("SESSIONE 2 (nuova sessione, stessa memoria)")
print("=" * 50)

history = carica_storia()  # Ricarica dal file

r = chat_persistente("Come mi chiamo? Ricordi cosa stavo studiando?")
print(f"🤖 {r}")

SESSIONE 2 (nuova sessione, stessa memoria)
📂 Storia caricata: 4 messaggi precedenti
💾 Storia salvata: 6 messaggi
🤖 Certo! 😊

Ti chiami **Luca** e studi **AI engineering a Sassari**.

L'ho ricordato dalla nostra conversazione appena iniziata. Però voglio essere trasparente: ricordo solo quello che mi hai detto *in questa conversazione specifica*. Se chiudessimo la chat e ricominciassimo domani, non avrei memoria di te - ogni conversazione è indipendente per me.

È una limitazione importante da sapere se pensi di lavorare con me regolarmente su progetti! 📝

C'è qualcosa di specifico su cui vuoi che ti aiuti?


---
## ⭐ Esercizi

In [ ]:
NOME_STUDENTE = "Giulio"  # ← SCRIVI IL TUO NOME
if NOME_STUDENTE:
    print(f"✅ Notebook di: {NOME_STUDENTE}")
else:
    print("⚠️ Scrivi il tuo nome!")

✅ Notebook di: Giulio


### Esercizio 1 — Chatbot multi-turno base ★☆☆
Crea un chatbot con system prompt WiData che mantiene la history. Fai almeno 4 domande collegate e verifica che risponda in modo coerente con i messaggi precedenti.

In [ ]:
# ESERCIZIO 1
history = []

system_widata = """Sei il supporto clienti di WiData Srl.
Devi rispondere solo su prodotti IoT, essere professionale,
rimandare al commerciale per prezzi e rifiutare domande fuori tema."""

domanda1 = chat("Quali prodotti IoT offrite per il monitoraggio industriale?", system=system_widata)
print("Domanda 1:", domanda1)

domanda2 = chat("Tra questi, quale consigliate per controllare la temperatura in un magazzino?")
print("Domanda 2:", domanda2)

domanda3 = chat("Supporta anche l'invio di alert in tempo reale via SMS o email?")
print("Domanda 3:", domanda3)

domanda4 = chat("Quanto costa il dispositivo che mi hai consigliato?")
print("Domanda 4:", domanda4)

print(history)

print(f"📊 Messaggi in history: {len(history)}")

Domanda 1: # Prodotti IoT per Monitoraggio Industriale - WiData

Grazie per la domanda! WiData offre una gamma completa di soluzioni IoT per il monitoraggio industriale:

## Principali categorie:

**Sensori e dispositivi di acquisizione dati:**
- Sensori di temperatura, umidità e pressione
- Sensori di vibrazione e accelerometri
- Sensori di corrente e consumi energetici
- Sensori di posizione e movimento

**Gateway e concentratori dati:**
- Gateway IoT per la raccolta e trasmissione dati
- Concentratori con connettività multi-protocollo (WiFi, Bluetooth, LoRaWAN, 4G/5G)

**Piattaforme di gestione:**
- Dashboard per visualizzazione real-time
- Sistemi di alert e notifiche
- Analitiche avanzate per la manutenzione predittiva

**Soluzioni specifiche per:**
- Monitoraggio consumi energetici
- Controllo qualità produttivo
- Tracciabilità e logistica
- Sicurezza impianti

## Come posso aiutarti?

Per informazioni dettagliate su una soluzione specifica, caratteristiche tecniche o per ricever

### Esercizio 2 — Sliding Window ★★☆
Implementa la sliding window: mantieni sempre il system prompt + gli ultimi 4 turni (8 messaggi). Testa che dopo 6 turni il chatbot non ricordi più i primi messaggi ma ricordi gli ultimi.

In [ ]:
history = []
MAX_TURNS = 4

def chat_sliding_window(messaggio, system=None):
    history.append({"role": "user", "content": messaggio})

    # Mantieni solo gli ultimi 4 turni = 8 messaggi
    history_window = history[-MAX_TURNS * 2:]

    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 500,
        "messages": history_window
    }

    if system:
        params["system"] = system

    risposta = client.messages.create(**params)
    testo = risposta.content[0].text

    history.append({"role": "assistant", "content": testo})

    # Taglia la history globale agli ultimi 8 messaggi
    if len(history) > MAX_TURNS * 2:
        del history[:-MAX_TURNS * 2]

    return testo

history = []

print(chat_sliding_window("Mi chiamo Alice"))
print(chat_sliding_window("Vivo a Milano"))
print(chat_sliding_window("Il mio colore preferito è il blu"))
print(chat_sliding_window("Lavoro come data analyst"))
print(chat_sliding_window("Mi piace Python"))
print(chat_sliding_window("Ho un cane che si chiama Luna"))

print(chat_sliding_window("Come mi chiamo?"))
print(chat_sliding_window("Come si chiama il mio cane?"))

Ciao Alice! 👋 Piacere di conoscerti. Come stai? Come posso aiutarti oggi?
Che bello! Milano è una città affascinante, con una ricca storia, una grande scena culturale e artistica. 🏙️

Ci sono tante cose interessanti da visitare e fare lì - dai Navigli al Duomo, dalla Pinacoteca di Brera ai musei. 

Abiti a Milano da molto tempo, o è un trasloco recente? C'è qualcosa in particolare in cui posso aiutarti?
Il blu è un colore bellissimo! 💙 È un colore che trasmette calma, serenità e fiducia. 

Ci sono tante sfumature di blu - dal blu cielo al blu mare, dal blu navy più scuro al celeste più chiaro. Ognuna ha il suo fascino.

Con una città come Milano e il blu come colore preferito, immagino che tu apprezzi paesaggi e ambienti armoniosi. Ti piace in particolare qualche sfumatura di blu, o le ami tutte?
Interessante! 📊 Il lavoro di data analyst è molto richiesto, soprattutto in una città come Milano che ha un forte settore tecnologico e finanziario.

Analizzare dati, trarre insights significa

### Esercizio 3 — Chatbot con streaming ★★☆
Riscrivi la funzione `chat()` usando lo streaming. L'output deve apparire parola per parola nel terminale. Aggiungi anche il conteggio dei token al termine.

In [ ]:
# ESERCIZIO 3

def chat_streaming(messaggio, history, system=None):
    history.append({"role": "user", "content": messaggio})

    full_text = ""

    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 300,
        "messages": history
    }

    if system:
        params["system"] = system

    with client.messages.stream(**params) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
            full_text += text

        final_message = stream.get_final_message()

    print()

    history.append({"role": "assistant", "content": full_text})

    input_tokens = final_message.usage.input_tokens
    output_tokens = final_message.usage.output_tokens
    total_tokens = input_tokens + output_tokens

    print(f"Token input: {input_tokens}")
    print(f"Token output: {output_tokens}")
    print(f"Token totali: {total_tokens}")

    return full_text

# Test
history = []
chat_streaming("Spiegami RAG in 3 frasi", history)

# RAG (Retrieval-Augmented Generation) in 3 frasi

1. **RAG combina ricerca e generazione**: un sistema che prima recupera documenti rilevanti da una base di dati, poi li usa per generare risposte più accurate e fondate.

2. **Risolve i limiti dei modelli isolati**: evita allucinazioni (risposte inventate) e fornisce informazioni aggiornate o specifiche del dominio che il modello di IA da solo non conosce.

3. **Caso d'uso pratico**: è come avere un assistente che legge i tuoi documenti aziendali prima di rispondere, invece di rispondere solo da memoria.
Token input: 19
Token output: 172
Token totali: 191


"# RAG (Retrieval-Augmented Generation) in 3 frasi\n\n1. **RAG combina ricerca e generazione**: un sistema che prima recupera documenti rilevanti da una base di dati, poi li usa per generare risposte più accurate e fondate.\n\n2. **Risolve i limiti dei modelli isolati**: evita allucinazioni (risposte inventate) e fornisce informazioni aggiornate o specifiche del dominio che il modello di IA da solo non conosce.\n\n3. **Caso d'uso pratico**: è come avere un assistente che legge i tuoi documenti aziendali prima di rispondere, invece di rispondere solo da memoria."

### Esercizio 4 — Chatbot con memoria persistente ★★★ (Deliverable!)

Costruisci il chatbot completo `chatbot_cli.py` con:
- History multi-turno
- Sliding window (max 10 messaggi)
- Streaming
- Memoria su JSON persistente
- System prompt WiData
- Loop interattivo con `input()` (digita 'esci' per uscire)

In [ ]:
# ESERCIZIO 4 — Chatbot completo (DELIVERABLE)
# Scrivi qui il codice completo del chatbot

import anthropic, json, os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

MEMORY_FILE = "chatbot_widata.json"
MAX_MESSAGGI = 10

SYSTEM = """
# ← Scrivi qui il tuo system prompt WiData
"""

def carica_storia():
    pass  # ← implementa

def salva_storia(history):
    pass  # ← implementa

def chat(messaggio, history):
    pass  # ← implementa con streaming + sliding window

# Loop principale
def main():
    history = carica_storia()
    print("🤖 Chatbot WiData avviato. Digita 'esci' per uscire.\n")

    while True:
        utente = input("Tu: ")
        if utente.lower() == "esci":
            print("👋 Arrivederci!")
            break
        risposta = chat(utente, history)
        # Lo streaming stampa già durante l'esecuzione

# main()  # Decommentare per eseguire

---
## 📤 Consegna

1. Completa tutti gli esercizi
2. Scarica: `File → Scarica → .ipynb`
3. Rinomina: `Lezione3_TUONOME.ipynb`
4. Carica su GitHub in `lezione3/`

```bash
git add lezione3/
git commit -m "Lezione 3 completata"
git push
```

---
### 📖 Per la prossima lezione (Giovedì 28/05)
Leggi **Huyen Cap. 6 — sezione RAG**

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*